# Stop guessing: improve prompts with a tiny eval set

**The Agent Build Log · Build #014 · Prompt Engineering 6/8**

Builds #009-#013 improved one ticket-triage prompt by looking at its mistakes and fixing them. This build makes that
loop explicit, with three ticket sets and four prompt versions, and asks the questions people usually skip: did the
fix generalise, how much of the gain is noise, and what does overfitting a prompt look like?

1. [Three sets with three jobs](#1-three-sets-with-three-jobs)
2. [Four prompt versions](#2-four-prompt-versions)
3. [The eval harness](#3-the-eval-harness)
4. [Scores by version and set](#4-scores-by-version-and-set)
5. [How much is noise?](#5-how-much-is-noise)
6. [Ticket by ticket: what each version fixed and broke](#6-ticket-by-ticket-what-each-version-fixed-and-broke)
7. [Findings](#7-findings)

**Model:** Claude Haiku 5.5, thinking off. **Needs:** an Anthropic API key. A full run costs about **6 cents**.

In [1]:
import json, os, random, re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
import anthropic

load_dotenv(Path.cwd() / ".env") or load_dotenv(Path.cwd().parents[1] / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in a .env file (see README)"
pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 80)

MODEL = "claude-haiku-5-5"
PRICE = {"input": 0.10, "output": 0.50}
client = anthropic.Anthropic().with_options(max_retries=6)
TK = json.loads(Path("data/tickets.json").read_text())
FIELDS = TK["fields"]
SETS = {"dev": TK["dev"], "validation": TK["validation"], "test": TK["test"]}
print({k: len(v) for k, v in SETS.items()})

{'dev': 20, 'validation': 20, 'test': 20}


## 1. Three sets with three jobs

| Set | Tickets | Job |
|---|---|---|
| **dev** | build #004's 20 | where you look at mistakes and tune |
| **validation** | build #010's 20 fresh ones | checks a fix generalises; v3's two new rules came from its misses, so for v3 and v4 it's no longer clean |
| **test** | 20 new tickets, labelled with rubric v3 **before any run** | touched once per version, never used to tune |

The labels are mine, so every score is agreement with my labelling rules.

In [2]:
pd.DataFrame([{**t["label"], "ticket": t["text"]} for t in TK["test"][:8]])

,category,priority,sentiment,ticket
0,account,urgent,negative,"Since the update, no one in our office can log in. We have 40 people waiting."
1,billing,medium,neutral,I was charged for express shipping on ORD-77310 but it came standard. Can I ...
2,feature_request,low,positive,"Your support team is amazing, thanks for fixing my order ORD-22014 so fast!"
3,account,high,negative,"The 'forgot password' link goes to a 404 page, so I can't get into my account."
4,feature_request,low,neutral,Would love a way to schedule reports to arrive every Monday.
5,shipping,medium,neutral,Parcel ORD-60331 says 'out for delivery' since Friday. Any news?
6,bug,high,negative,The invoice PDF shows last year's address. Our finance team needs it fixed b...
7,account,urgent,negative,Someone changed my password and my email address. I didn't do this!


## 2. Four prompt versions

- **v1:** build #009's rubric.
- **v2:** + the three tie-breaks from build #010.
- **v3:** + two rules for the validation misses builds #010-#011 kept hitting: a login problem is *account* even when a bug
  causes it, and praise is *feature_request* even when it mentions an order or refund.
- **v4 (overfitted on purpose):** v3 + four rules written for the exact dev tickets v1 got wrong
  ("questions about changing an account email are neutral", "requests for webhooks are positive"…). The kind of rule
  you add when you tune by staring at one set.

In [3]:
ROLE = "You are the support triage lead at an online electronics shop. You sort incoming tickets so the right team picks them up first."
V1 = '''How our support team labels tickets:
Category
- billing: charges, refunds, plans, pricing questions, cancellations
- bug: something in the product is broken or wrong
- account: logging in, security, account settings, users
- shipping: delivery, tracking, damaged or wrong items
- feature_request: ideas for the product, and general feedback or praise
Priority
- urgent: many users blocked or money being lost right now, or a security incident (someone else in the account)
- high: one customer blocked with no workaround (locked out, charged twice for the same order, item lost or damaged,
  a crash on a main screen), or a business deadline at risk
- medium: a real problem with a workaround or a simple fix (wrong amount, coupon, cancellation, late tracking,
  exchange, a non-blocking bug), or a feature they would use right away
- low: questions, "someday" ideas, praise
Sentiment
- negative: frustration, or reports something broken, lost, damaged or blocked for them
- neutral: a calm request or question, including calm billing requests
- positive: praise or thanks is the main point'''   # build #009
TIE_BREAKS = '''When two rules disagree:
- A calm request about a billing mistake is neutral, even though something went wrong.
- A bug the customer calls minor or small is still negative: it reports something broken.
- If the customer has a working alternative (another device, the website), it's medium, even if one path is blocked.'''   # build #010
V3_RULES = '''- Login, session and password problems are account, even when a bug causes them.
- Praise or thanks is feature_request, even when it mentions an order or a refund.'''
V4_HACKS = '''- Questions about changing an account email are neutral.
- Requests for webhooks are positive.
- Tracking that hasn't updated for days is negative.
- Wrong VAT numbers on invoices are bugs.'''

VERSIONS = {
    "v1": V1,
    "v2": V1 + "\n" + TIE_BREAKS,
    "v3": V1 + "\n" + TIE_BREAKS + "\n" + V3_RULES,
    "v4 (overfitted)": V1 + "\n" + TIE_BREAKS + "\n" + V3_RULES + "\n" + V4_HACKS,
}
SCHEMA = ('{"category": one of ' + json.dumps(FIELDS["category"]) + ', "priority": one of ' + json.dumps(FIELDS["priority"])
          + ', "sentiment": one of ' + json.dumps(FIELDS["sentiment"]) + "}")
print(VERSIONS["v4 (overfitted)"][-700:])

or thanks is the main point
When two rules disagree:
- A calm request about a billing mistake is neutral, even though something went wrong.
- A bug the customer calls minor or small is still negative: it reports something broken.
- If the customer has a working alternative (another device, the website), it's medium, even if one path is blocked.
- Login, session and password problems are account, even when a bug causes them.
- Praise or thanks is feature_request, even when it mentions an order or a refund.
- Questions about changing an account email are neutral.
- Requests for webhooks are positive.
- Tracking that hasn't updated for days is negative.
- Wrong VAT numbers on invoices are bugs.


## 3. The eval harness

Twenty lines that are the whole method: run a prompt version over a set, 3 times per ticket, and keep every answer.

In [4]:
def classify(rubric, ticket):
    user = (f"{rubric}\n\nTriage this support ticket: give its category, priority and sentiment.\n\n<ticket>\n{ticket}\n</ticket>\n\n"
            f"Reply with only a JSON object, no other text:\n{SCHEMA}")
    r = client.messages.create(model=MODEL, max_tokens=300, thinking={"type": "disabled"}, system=ROLE,
                               messages=[{"role": "user", "content": user}])
    text = "".join(b.text for b in r.content if b.type == "text")
    m = re.search(r"\{.*?\}", text, re.S)
    try:
        d = json.loads(m.group(0)) if m else {}
    except json.JSONDecodeError:
        d = {}
    got = {f: str(d.get(f, "")).lower() for f in FIELDS}
    return got, (r.usage.input_tokens * PRICE["input"] + r.usage.output_tokens * PRICE["output"]) / 1e6

def evaluate(version, split, runs=3):
    jobs = [(t, run) for t in SETS[split] for run in range(1, runs + 1)]
    with ThreadPoolExecutor(8) as ex:
        outs = list(ex.map(lambda j: (j, classify(VERSIONS[version], j[0]["text"])), jobs))
    return [{"version": version, "set": split, "id": t["id"], "run": run, "correct": got == t["label"],
             **{f"got_{f}": got[f] for f in FIELDS}, "cost": c} for (t, run), (got, c) in outs]

rows = [r for v in VERSIONS for s in SETS for r in evaluate(v, s)]
ev = pd.DataFrame(rows)
print(f"{len(ev)} calls | ${ev['cost'].sum():.4f}")

720 calls | $0.0589


## 4. Scores by version and set

All 3 labels right, % of runs (60 per cell).

In [5]:
scores = (100 * ev.pivot_table(index="version", columns="set", values="correct", aggfunc="mean")[list(SETS)]).round().astype(int)
scores = scores.loc[list(VERSIONS)]
scores["dev - test gap"] = scores["dev"] - scores["test"]
scores

set,dev,validation,test,dev - test gap
version,,,,
v1,85,73,70,15
v2,87,87,85,2
v3,90,98,90,0
v4 (overfitted),100,93,95,5


## 5. How much is noise?

A test score from 20 tickets moves by 5 points per ticket. Resampling the tickets (bootstrap, 2,000 times) gives a range
for each version's test score: if two ranges overlap a lot, the eval can't tell those versions apart.

In [6]:
def per_ticket(version, split):
    return ev[(ev.version == version) & (ev.set == split)].groupby("id")["correct"].mean()

rng = random.Random(0)
def bootstrap(series, n=2000):
    vals = list(series.values)
    means = sorted(sum(rng.choice(vals) for _ in vals) / len(vals) for _ in range(n))
    return round(100 * means[int(0.025 * n)]), round(100 * means[int(0.975 * n)])

ci = pd.DataFrame([{"version": v, "test score %": round(100 * per_ticket(v, "test").mean()),
                    "95% range": "%d-%d%%" % bootstrap(per_ticket(v, "test"))} for v in VERSIONS])
ci

,version,test score %,95% range
0,v1,70,50-90%
1,v2,85,70-100%
2,v3,90,75-100%
3,v4 (overfitted),95,85-100%


## 6. Ticket by ticket: what each version fixed and broke

An average hides trades. For each step from one version to the next, count the test tickets that got better
(more of their 3 runs right) and worse.

In [7]:
steps = [("v1", "v2"), ("v2", "v3"), ("v3", "v4 (overfitted)")]
moves = []
for a, b in steps:
    for split in SETS:
        d = per_ticket(b, split) - per_ticket(a, split)
        moves.append({"step": f"{a} → {b}", "set": split, "tickets better": int((d > 0).sum()), "tickets worse": int((d < 0).sum()),
                      "net runs": int(round(3 * d.sum()))})
moves = pd.DataFrame(moves)
moves

,step,set,tickets better,tickets worse,net runs
0,v1 → v2,dev,1,0,1
1,v1 → v2,validation,4,1,8
2,v1 → v2,test,3,0,9
3,v2 → v3,dev,1,0,2
4,v2 → v3,validation,3,1,7
5,v2 → v3,test,2,1,3
6,v3 → v4 (overfitted),dev,2,0,6
7,v3 → v4 (overfitted),validation,1,2,-3
8,v3 → v4 (overfitted),test,1,0,3


In [8]:
# test tickets that no version gets fully right
text_of = {t["id"]: t for t in TK["test"]}
hard = pd.DataFrame({v: per_ticket(v, "test") for v in VERSIONS})
hard = hard[(hard < 1).any(axis=1)]
hard.insert(0, "label", [text_of[i]["label"] for i in hard.index])
hard.insert(0, "ticket", [text_of[i]["text"][:60] for i in hard.index])
hard.round(2)

,ticket,label,v1,v2,v3,v4 (overfitted)
id,,,,,,
403,"Your support team is amazing, thanks for fixing my order ORD","{'category': 'feature_request', 'priority': 'low', 'sentiment': 'positive'}",0.0,0.0,1.0,1.0
404,"The 'forgot password' link goes to a 404 page, so I can't ge","{'category': 'account', 'priority': 'high', 'sentiment': 'negative'}",0.0,0.0,1.0,1.0
405,Would love a way to schedule reports to arrive every Monday.,"{'category': 'feature_request', 'priority': 'low', 'sentiment': 'neutral'}",0.0,1.0,0.0,1.0
407,The invoice PDF shows last year's address. Our finance team,"{'category': 'bug', 'priority': 'high', 'sentiment': 'negative'}",0.0,0.0,0.0,0.0
414,The app freezes on the checkout screen on my Android phone.,"{'category': 'bug', 'priority': 'medium', 'sentiment': 'negative'}",0.0,1.0,1.0,1.0
419,Please add my accountant as a read-only user on our account.,"{'category': 'account', 'priority': 'low', 'sentiment': 'neutral'}",0.0,1.0,1.0,1.0


## 7. Findings

Every number below comes from the cells above; these are the numbers the LinkedIn post uses.

In [9]:
f = {
    "model": MODEL, "sdk": anthropic.__version__, "thinking": "disabled", "calls": len(ev), "cost": round(ev["cost"].sum(), 4),
    "scores": scores.reset_index().to_dict("records"), "test_ci": ci.to_dict("records"), "moves": moves.to_dict("records"),
    "hard_test": hard.reset_index().to_dict("records"), "runs": ev.to_dict("records"),
}
Path("findings.json").write_text(json.dumps(f, indent=2, default=str))
print(json.dumps({k: f[k] for k in ("scores", "test_ci", "cost")}, indent=1, default=str))

{
 "scores": [
  {
   "version": "v1",
   "dev": 85,
   "validation": 73,
   "test": 70,
   "dev - test gap": 15
  },
  {
   "version": "v2",
   "dev": 87,
   "validation": 87,
   "test": 85,
   "dev - test gap": 2
  },
  {
   "version": "v3",
   "dev": 90,
   "validation": 98,
   "test": 90,
   "dev - test gap": 0
  },
  {
   "version": "v4 (overfitted)",
   "dev": 100,
   "validation": 93,
   "test": 95,
   "dev - test gap": 5
  }
 ],
 "test_ci": [
  {
   "version": "v1",
   "test score %": 70,
   "95% range": "50-90%"
  },
  {
   "version": "v2",
   "test score %": 85,
   "95% range": "70-100%"
  },
  {
   "version": "v3",
   "test score %": 90,
   "95% range": "75-100%"
  },
  {
   "version": "v4 (overfitted)",
   "test score %": 95,
   "95% range": "85-100%"
  }
 ],
 "cost": 0.0589
}
